In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests
import itertools
import string

In [ ]:
# Classification and Labeling of copy number status at gene loci for oncoprint
def classify_copy_number(row):
    # 1. Extract values
    a1 = float(row['rescaled.cn.a1'])
    a2 = float(row['rescaled.cn.a2'])
    ploidy = float(row['ploidy'])

    # Calculate derived metrics
    total_cn = a1 + a2
    minor_allele = min(a1, a2)
    
    # Tolerances
    eps = 0.25      # Tolerance for exact zero/balance
    delta = 0.5     # Threshold for Gain/Loss against fractional ploidy
    diploid = 2.0   # Value for diploid genome
    # Anchor baseline to 2.0 (Human normal) so sub-diploid tumors don't skew logic
    baseline = max(2.0, ploidy)
    
    # Boolean states
    is_loh = minor_allele < eps
    is_balanced = abs(a1 - a2) < eps

    # TIER 1: EXTREME EVENTS
    if total_cn < 0.25:
        return "Deep Deletion"

    # Rule: > 2x Ploidy AND > 4 copies
    if total_cn >= (2 * ploidy) and total_cn >= 4.0:
        return "Amplification"


    # TIER 2: LOSSES (Catches 1+0 before CN-LOH can mislabel it)
    # For ploidy 1.9, total CN 1.0 -> 1.0 <= (2.0 - 0.5) -> True -> Het D.
    if total_cn <= (baseline - delta):
        # exactly back to diploid state (so no loss relative to diploid cells status, but relative to Ploidy)
        # The next logic works because only samples were total_cn is smaller then ploidy (baseline if amplified) are submitted to this if clause
        if abs(total_cn - diploid) <= eps:
            return "Diploid and LOH after Post-WGD-Loss" if is_loh else "Diploid after Post-WGD-loss"
        # exactly back to 
        elif total_cn < (diploid - eps):
            if ploidy >= 2.5:
                return "Severe Post-WGD loss (LOH)" if is_loh else "Severe Post-WGD Loss"
            else:
                return "Het. D. (LOH)" if is_loh else "Het. D."
            
        # Case C: Relative Loss, but still above diploid (e.g., 1+2 = 3 in ploidy 4)
        else:
            return "Post-WGD Loss (LOH)" if is_loh else "Post-WGD Loss"
        


    # TIER 3: COPY-NEUTRAL LOH
    # We lost an allele, but total copies remain at the baseline
    if is_loh and abs(total_cn - baseline) < delta:
        if abs(total_cn- diploid) <= eps:
            return "CN-LOH"
        else:
            return "Polyploid LOH (Absolute Gain)"


    # TIER 4: GAINS & WGD
    # Scenario A: WGD Passenger (Strictly balanced, matches ploidy)
    # e.g., 2+2 in a 3.9 ploidy tumor
    if ploidy >= 2.5 and is_balanced and abs(total_cn - ploidy) < delta:
        return "WGD (N)"
            
    # Scenario B: True Gain
    # e.g., 1+3 (Total 4) against a 3.36 ploidy -> 4 >= 3.86 -> True!
    if total_cn >= (diploid + delta):
        if total_cn >= (ploidy + delta +0.25):
            return "Focal Gain"
        if total_cn <= ploidy:
            return "Absolute Gain/Relative Loss"

    # TIER 5: NEUTRAL
    return "Neutral"
    

def handle_merges(cell_value):
    # 1. Handle actual NaNs or empty values immediately
    if pd.isna(cell_value) or cell_value == '' or cell_value == 'nan':
        return pd.Series([np.nan, np.nan])
        
    # 2. Force to string to prevent 'float' errors
    val_str = str(cell_value)
    
    # 3. Count mutations based on commas
    parts = [p.strip() for p in val_str.split(',') if p.strip()]
        
    # CASE 1: 3 or more mutations (The "Merged/Union" Case)
    if len(parts) >= 3:
        return pd.Series([val_str, np.nan])
        
    # CASE 2: Exactly 2 mutations
    if len(parts) == 2:
        return pd.Series([parts[0], parts[1]])
        
    # CASE 3: Single mutation
    if len(parts) == 1:
        return pd.Series([parts[0], np.nan])
        
    return pd.Series([np.nan, np.nan])


def split_BNDs_position_fast(df):
    """Pair the two breakend rows of each BND 'id' into one chr1/pos1/chr2/pos2 record."""
    df = df.sort_values(['id', 'pos1']).copy()

    counts = df.groupby('id')['id'].transform('size')
    mask = (counts == 2)

    next_pos = df['pos1'].shift(-1)
    prev_pos = df['pos1'].shift(1)
    next_chr = df['chr1'].shift(-1)
    prev_chr = df['chr1'].shift(1)

    is_first_in_pair = (df['id'] == df['id'].shift(-1)) & mask
    is_second_in_pair = (df['id'] == df['id'].shift(1)) & mask

    df.loc[is_first_in_pair, 'pos2'] = next_pos
    df.loc[is_first_in_pair, 'chr2'] = next_chr
    df.loc[is_second_in_pair, 'pos2'] = prev_pos
    df.loc[is_second_in_pair, 'chr2'] = prev_chr

    # Singleton BND (no pair): assume chr2 == chr1
    df['chr2'] = df['chr2'].fillna(df['chr1']).astype(int)
    return df


#  Tumor-Sampel-Barcode is a merge of Tumor-ID and Normal-ID, this helps to seperate them again if necessary e.g to identify multiple tumor sampels from same patient (same normal)
def extract_normal_id(df):
    """extract normal ID from Tumor_Sample_Name"""

    parts = df.split('-')
    # If there is only one hyphen, the normal is simply the rigth side
    if len(parts) == 2:
        return parts[1]
    else:
        # for TARGET samples with multiple hyphens
        mid = len(parts) // 2
        return "-".join(parts[mid:])


#  Tumor-Sampel-Barcode is a merge of Tumor-ID and Normal-ID, this helps to seperate them again if necessary (as some inputs only have e.g. Tumor ID)
def extract_tumor_id(df):
    """extract normal ID from Tumor_Sample_Name"""

    parts = df.split('-')
    # If there is only one hyphen, the normal is simply the rigth side
    if len(parts) == 2:
        return parts[0]
    else:
        # for TARGET samples with multiple hyphens
        mid = len(parts) // 2
        return "-".join(parts[:mid])

    
# Function for generation of alphabetical labels 
def label_generator():
    # if count is 1
    for size in itertools.count(1):
        for chars in itertools.product(string.ascii_lowercase, repeat=size):
            yield "".join(chars)

### Absolute Copy Number - Gene wise Classification (required for oncoprint but not for other analyses)

In [3]:
Abs_gene_allelic = pd.read_csv('../../../CNVs_OS/Absolute/gene_allelic_cn_v2.tsv', sep='\t')
Abs_gene_allelic

,gene,gene_start,gene_end,chrom,seg_start,seg_end,cancer.cell.frac.a1,cancer.cell.frac.a2,LOH,HZ,SC_HZ,rescaled.cn.a1,rescaled.cn.a2,name
0,SAMD11,923923,944575,1,832000.0,3947999,0.12,0.16,0,0,0,2.0,2.0,09T02-09N01
1,NOC2L,944203,959309,1,832000.0,3947999,0.12,0.16,0,0,0,2.0,2.0,09T02-09N01
2,KLHL17,960584,965719,1,832000.0,3947999,0.12,0.16,0,0,0,2.0,2.0,09T02-09N01
3,PLEKHN1,966482,975865,1,832000.0,3947999,0.12,0.16,0,0,0,2.0,2.0,09T02-09N01
4,PERM1,975198,982117,1,832000.0,3947999,0.12,0.16,0,0,0,2.0,2.0,09T02-09N01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4714492,CHKB,50578959,50601455,22,43376000.0,50759999,0.29,0.23,0,0,0,1.0,1.0,WGS_15_2-WGS_16_2
4714493,MAPK8IP2,50600793,50613981,22,43376000.0,50759999,0.29,0.23,0,0,0,1.0,1.0,WGS_15_2-WGS_16_2
4714494,ARSA,50622754,50628173,22,43376000.0,50759999,0.29,0.23,0,0,0,1.0,1.0,WGS_15_2-WGS_16_2
4714495,SHANK3,50674408,50733212,22,43376000.0,50759999,0.29,0.23,0,0,0,1.0,1.0,WGS_15_2-WGS_16_2


In [ ]:
metadata = pd.read_csv('../../../metadata/metadata_ABSOLUTE_Annotation_Ploidy.tsv', sep='\t')
metadata[['Tumor_Sample_Barcode', 'purity', 'WGD_Category', 'Union_Sample_Name' ]]

,Tumor_Sample_Barcode,purity,WGD_Category,Union_Sample_Name
0,09T02-09N01,0.74,wgd,09T02-09N01-MT-union
1,09T03-09N01,0.33,wgd,09T02-09N01-MT-union
2,10T03-10N02,0.56,diploid,10T03-10N02
3,13T02-13N02,0.56,wgd,13T02-13N02
4,17_439_00067_T2-17_439_00067_WB,0.83,diploid,17_439_00067_T2-17_439_00067_WB
...,...,...,...,...
231,WGS_12_15-WGS_11_13,0.90,diploid,WGS_12_15-WGS_11_13
232,WGS_12_19-WGS_11_17,0.76,diploid,WGS_12_19-WGS_11_17
233,WGS_12_31-WGS_11_29,0.30,wgd,WGS_12_31-WGS_11_29
234,WGS_12_7-WGS_11_5,0.73,wgd,WGS_12_7-WGS_11_5


In [5]:
# Merge the ploidy info from ABSOLUTE (in metadata file) output into the gene alllic absolute file
annotated_genes = pd.merge(
    Abs_gene_allelic,
    metadata[['Tumor_Sample_Barcode', 'ploidy']],
    left_on='name',
    right_on='Tumor_Sample_Barcode',
    how='left'
)
annotated_genes

,gene,gene_start,gene_end,chrom,seg_start,seg_end,cancer.cell.frac.a1,cancer.cell.frac.a2,LOH,HZ,SC_HZ,rescaled.cn.a1,rescaled.cn.a2,name,Tumor_Sample_Barcode,ploidy
0,SAMD11,923923,944575,1,832000.0,3947999,0.12,0.16,0,0,0,2.0,2.0,09T02-09N01,09T02-09N01,3.36
1,NOC2L,944203,959309,1,832000.0,3947999,0.12,0.16,0,0,0,2.0,2.0,09T02-09N01,09T02-09N01,3.36
2,KLHL17,960584,965719,1,832000.0,3947999,0.12,0.16,0,0,0,2.0,2.0,09T02-09N01,09T02-09N01,3.36
3,PLEKHN1,966482,975865,1,832000.0,3947999,0.12,0.16,0,0,0,2.0,2.0,09T02-09N01,09T02-09N01,3.36
4,PERM1,975198,982117,1,832000.0,3947999,0.12,0.16,0,0,0,2.0,2.0,09T02-09N01,09T02-09N01,3.36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4714492,CHKB,50578959,50601455,22,43376000.0,50759999,0.29,0.23,0,0,0,1.0,1.0,WGS_15_2-WGS_16_2,WGS_15_2-WGS_16_2,1.90
4714493,MAPK8IP2,50600793,50613981,22,43376000.0,50759999,0.29,0.23,0,0,0,1.0,1.0,WGS_15_2-WGS_16_2,WGS_15_2-WGS_16_2,1.90
4714494,ARSA,50622754,50628173,22,43376000.0,50759999,0.29,0.23,0,0,0,1.0,1.0,WGS_15_2-WGS_16_2,WGS_15_2-WGS_16_2,1.90
4714495,SHANK3,50674408,50733212,22,43376000.0,50759999,0.29,0.23,0,0,0,1.0,1.0,WGS_15_2-WGS_16_2,WGS_15_2-WGS_16_2,1.90


In [ ]:
# Classification of Copy number changes. def classify_copy_number applied
# Ensure columns are floats to avoid errors
cols_to_fix = ['rescaled.cn.a1', 'rescaled.cn.a2', 'ploidy']
annotated_genes[cols_to_fix] = annotated_genes[cols_to_fix].astype(float)

# Run the classification
annotated_genes['cn_status'] = annotated_genes.apply(classify_copy_number, axis=1)

suspects = ['TP53', 'MYC', 'MDM2', 'CDK4', 'RB1', 'CCND3','CCNE1', 'TOP3A', 'NF2', 'PTEN', 'PVT1', 'AKT1', 'AKT2', 'CDC5L', 'CDKN2A', 'CDKN2B', 'CDKN1A', 'CDKN1B', 'RUNX2', 'NSD1']
annotated_genes[annotated_genes['gene'].isin(suspects)][['Tumor_Sample_Barcode', 'gene', 'rescaled.cn.a1', 'rescaled.cn.a2', 'ploidy', 'cn_status']]

,Tumor_Sample_Barcode,gene,rescaled.cn.a1,rescaled.cn.a2,ploidy,cn_status
5996,09T02-09N01,NSD1,1.0,2.0,3.36,Absolute Gain/Relative Loss
6496,09T02-09N01,CDKN1A,1.0,2.0,3.36,Absolute Gain/Relative Loss
6548,09T02-09N01,CCND3,1.0,2.0,3.36,Absolute Gain/Relative Loss
6610,09T02-09N01,CDC5L,1.0,2.0,3.36,Absolute Gain/Relative Loss
6612,09T02-09N01,RUNX2,1.0,2.0,3.36,Absolute Gain/Relative Loss
...,...,...,...,...,...,...
4710548,WGS_15_2-WGS_16_2,TP53,0.0,1.0,1.90,Het. D. (LOH)
4710669,WGS_15_2-WGS_16_2,TOP3A,1.0,2.0,1.90,Focal Gain
4712461,WGS_15_2-WGS_16_2,CCNE1,1.0,2.0,1.90,Focal Gain
4712664,WGS_15_2-WGS_16_2,AKT2,1.0,1.0,1.90,Neutral


In [ ]:
# Remove duplicate entries prioritize if multiple changes occur for the same gene name (ABSOLUTE does segment and sometimes parts of a gene are on different segments with different or the same outcome (duplicates))
oncogene_severity = {
    'Amplification': 15,
    'Focal Gain': 14,
    'Polyploid LOH (Absolute Gain)': 13,
    'Absolute Gain/Relative Loss': 12,
    'WGD (N)': 11,
    'Neutral': 10,
    'CN-LOH': 9,
    'Diploid after Post-WGD-loss': 8,
    'Post-WGD Loss': 7,
    'Het. D.': 6,
    'Severe Post-WGD Loss': 5,
    'Post-WGD Loss (LOH)': 4,
    'Diploid and LOH after Post-WGD-Loss': 3,
    'Het. D. (LOH)': 2,
    'Severe Post-WGD loss (LOH)': 1,
    'Deep Deletion': 0
}

# Map for TSGs: Prioritize Deletion/Loss
tsg_severity = {
    'Deep Deletion': 15,
    'Severe Post-WGD loss (LOH)': 14,
    'Het. D. (LOH)': 13,
    'Diploid and LOH after Post-WGD-Loss': 12,
    'Post-WGD Loss (LOH)': 11,
    'Severe Post-WGD Loss': 10,
    'Het. D.': 9,
    'Post-WGD Loss': 8,
    'Diploid after Post-WGD-loss': 7,
    'CN-LOH': 6,
    'Neutral': 5,
    'WGD (N)': 4,
    'Absolute Gain/Relative Loss': 3,
    'Polyploid LOH (Absolute Gain)': 2,
    'Focal Gain': 1,
    'Amplification': 0
}

# Define list of TSGs
tsg_list = ["TP53", "CDKN2A", "RB1", "ATRX", "NF2", "PTEN", "CDKN2B", "CDKN1A", "CDKN1B", "NSD1"]

def get_severity(row):
    # Determine which map to use
    if row['gene'] in tsg_list:
        return tsg_severity.get(row['cn_status'], 0)
    else:
        return oncogene_severity.get(row['cn_status'], 0)

# 1. Apply the gene-specific severity score
annotated_genes['severity_score'] = annotated_genes.apply(get_severity, axis=1)

# 2. Sort: Sample -> Gene -> Severity (Highest First)
annotated_genes_sorted = annotated_genes.sort_values(
    by=['Tumor_Sample_Barcode', 'gene', 'severity_score'],
    ascending=[True, True, False]
)

# 3. Deduplicate to keep only the most biologically relevant segment per gene
annotated_genes_clean = annotated_genes_sorted.drop_duplicates(
    subset=['Tumor_Sample_Barcode', 'gene'],
    keep='first'
)



In [ ]:
# Just renaming the metadata columns which were only gene names to a better name that indicates if a sample has a clustered event occuring at a hotspot loci
metadata.rename(columns={'CCND3':'CCND3_CGR_int', 'MYC':'MYC_CGR_int', 'CDK4-MDM2':'CDK4-MDM2_CGR_int', 'CCNE1':'CCNE1_CGR_int', 'TP53':'TP53_CGR_int'}, inplace=True)
# Pivot output to annotate to metadata table
classification = annotated_genes_clean.pivot(columns='gene', index='Tumor_Sample_Barcode', values='cn_status')
classification

gene,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZUP1,ZW10,ZWILCH,ZWINT,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
Tumor_Sample_Barcode,,,,,,,,,,,,,,,,,,,,,
NaN,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,...,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral
09T02-09N01,Focal Gain,Diploid after Post-WGD-loss,Absolute Gain/Relative Loss,Absolute Gain/Relative Loss,Neutral,Absolute Gain/Relative Loss,Neutral,Absolute Gain/Relative Loss,Absolute Gain/Relative Loss,Neutral,...,Absolute Gain/Relative Loss,Absolute Gain/Relative Loss,Neutral,Diploid after Post-WGD-loss,Neutral,Neutral,Neutral,Absolute Gain/Relative Loss,Diploid and LOH after Post-WGD-Loss,Neutral
09T03-09N01,Neutral,Diploid after Post-WGD-loss,Absolute Gain/Relative Loss,Absolute Gain/Relative Loss,Neutral,Absolute Gain/Relative Loss,Neutral,Absolute Gain/Relative Loss,Absolute Gain/Relative Loss,Neutral,...,Absolute Gain/Relative Loss,Absolute Gain/Relative Loss,Absolute Gain/Relative Loss,Diploid after Post-WGD-loss,Neutral,Neutral,Neutral,Neutral,Diploid and LOH after Post-WGD-Loss,Neutral
10T03-10N02,Het. D. (LOH),Het. D. (LOH),Neutral,Neutral,CN-LOH,Het. D. (LOH),Het. D. (LOH),Het. D. (LOH),Neutral,Het. D. (LOH),...,Neutral,Neutral,Het. D. (LOH),Neutral,Het. D. (LOH),CN-LOH,CN-LOH,Neutral,Neutral,Focal Gain
13T02-13N02,Diploid and LOH after Post-WGD-Loss,Polyploid LOH (Absolute Gain),Focal Gain,Focal Gain,Focal Gain,Post-WGD Loss (LOH),Post-WGD Loss (LOH),Post-WGD Loss (LOH),Neutral,Post-WGD Loss (LOH),...,Post-WGD Loss (LOH),Neutral,Neutral,Polyploid LOH (Absolute Gain),Polyploid LOH (Absolute Gain),Focal Gain,Focal Gain,Focal Gain,Polyploid LOH (Absolute Gain),Focal Gain
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WGS_12_15-WGS_11_13,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Het. D. (LOH),...,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Het. D. (LOH),Neutral
WGS_12_19-WGS_11_17,Diploid after Post-WGD-loss,Diploid after Post-WGD-loss,Focal Gain,Focal Gain,Focal Gain,Neutral,Focal Gain,Neutral,Neutral,Focal Gain,...,Diploid and LOH after Post-WGD-Loss,Neutral,Diploid after Post-WGD-loss,Diploid after Post-WGD-loss,Focal Gain,Focal Gain,Focal Gain,Diploid and LOH after Post-WGD-Loss,Polyploid LOH (Absolute Gain),Focal Gain
WGS_12_31-WGS_11_29,Diploid and LOH after Post-WGD-Loss,Neutral,Amplification,Amplification,Neutral,Neutral,Absolute Gain/Relative Loss,Diploid and LOH after Post-WGD-Loss,Diploid and LOH after Post-WGD-Loss,Absolute Gain/Relative Loss,...,Neutral,Diploid after Post-WGD-loss,Diploid after Post-WGD-loss,Absolute Gain/Relative Loss,Diploid after Post-WGD-loss,Neutral,Neutral,Neutral,Neutral,Neutral


In [ ]:
#  Select the specific genes to be annotated 
target_genes = ['TP53', 'MYC', 'MDM2', 'CDK4', 'RB1', 'CCND3','CCNE1', 'PTEN', 'CDKN2A']

gene_annotations = classification[target_genes]

# Merge into metadata
meta_new_annotated = metadata.merge(
    gene_annotations,
    left_on='Tumor_Sample_Barcode',   
    right_index=True,                 
    how='left'                        
)

# If a sample was in metadata but not in the CNV table, fill with 'Not Available'
meta_new_annotated[target_genes] = meta_new_annotated[target_genes].fillna('Not_available')

# Preview the result
meta_new_annotated

,Unnamed: 0,Tumor_Sample_Barcode,participant,study,sex,Ethnicity,Age_at_diagnosis,Sample_type,Location_Primary_Site,Metastatis_at_diagnosis,...,Purity_Terra,TP53,MYC,MDM2,CDK4,RB1,CCND3,CCNE1,PTEN,CDKN2A
0,0,09T02-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,...,0.74,Diploid and LOH after Post-WGD-Loss,Neutral,Absolute Gain/Relative Loss,Absolute Gain/Relative Loss,Deep Deletion,Absolute Gain/Relative Loss,Amplification,Diploid after Post-WGD-loss,Neutral
1,1,09T03-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,...,0.33,Diploid and LOH after Post-WGD-Loss,Neutral,Absolute Gain/Relative Loss,Absolute Gain/Relative Loss,Deep Deletion,Absolute Gain/Relative Loss,Amplification,Diploid after Post-WGD-loss,Neutral
2,2,10T03-10N02,10,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,...,0.56,Het. D. (LOH),Amplification,Het. D. (LOH),Het. D. (LOH),Het. D. (LOH),Neutral,Neutral,Neutral,Neutral
3,3,13T02-13N02,13,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,...,0.56,Polyploid LOH (Absolute Gain),Focal Gain,Neutral,Post-WGD Loss,Diploid and LOH after Post-WGD-Loss,Amplification,Diploid and LOH after Post-WGD-Loss,Polyploid LOH (Absolute Gain),Deep Deletion
4,4,17_439_00067_T2-17_439_00067_WB,17_439_00067,DFCI_with_snRNAseq,Female,NaN,10.0,"Recurrent, Metastatic",NaN,No Metastasis,...,0.83,Diploid and LOH after Post-WGD-Loss,Focal Gain,Diploid after Post-WGD-loss,Diploid after Post-WGD-loss,Severe Post-WGD loss (LOH),Neutral,Focal Gain,Diploid and LOH after Post-WGD-Loss,Diploid and LOH after Post-WGD-Loss
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,231,WGS_12_15-WGS_11_13,participant_21,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,...,0.90,Het. D. (LOH),Not_available,Neutral,Neutral,Het. D. (LOH),Neutral,Neutral,Neutral,Neutral
232,232,WGS_12_19-WGS_11_17,participant_47,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,...,0.76,Polyploid LOH (Absolute Gain),Neutral,Neutral,Neutral,Severe Post-WGD loss (LOH),Focal Gain,Diploid after Post-WGD-loss,Diploid after Post-WGD-loss,Amplification
233,233,WGS_12_31-WGS_11_29,participant_31,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,...,0.30,Absolute Gain/Relative Loss,Neutral,Not_available,Not_available,Diploid and LOH after Post-WGD-Loss,Neutral,Neutral,Neutral,Diploid and LOH after Post-WGD-Loss
234,235,WGS_12_7-WGS_11_5,participant_37,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,...,0.73,Not_available,Amplification,Neutral,Polyploid LOH (Absolute Gain),Post-WGD Loss (LOH),Focal Gain,Post-WGD Loss (LOH),Absolute Gain/Relative Loss,Post-WGD Loss


In [ ]:
# Renamed column names for annotated genes
rename_mapping = {gene: f"{gene}_CNV_Status" for gene in target_genes}

# Apply it to dataframe
meta_new_annotated.rename(columns=rename_mapping, inplace=True)
meta_new_annotated

,Unnamed: 0,Tumor_Sample_Barcode,participant,study,sex,Ethnicity,Age_at_diagnosis,Sample_type,Location_Primary_Site,Metastatis_at_diagnosis,...,Purity_Terra,TP53_CNV_Status,MYC_CNV_Status,MDM2_CNV_Status,CDK4_CNV_Status,RB1_CNV_Status,CCND3_CNV_Status,CCNE1_CNV_Status,PTEN_CNV_Status,CDKN2A_CNV_Status
0,0,09T02-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,...,0.74,Diploid and LOH after Post-WGD-Loss,Neutral,Absolute Gain/Relative Loss,Absolute Gain/Relative Loss,Deep Deletion,Absolute Gain/Relative Loss,Amplification,Diploid after Post-WGD-loss,Neutral
1,1,09T03-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,...,0.33,Diploid and LOH after Post-WGD-Loss,Neutral,Absolute Gain/Relative Loss,Absolute Gain/Relative Loss,Deep Deletion,Absolute Gain/Relative Loss,Amplification,Diploid after Post-WGD-loss,Neutral
2,2,10T03-10N02,10,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,...,0.56,Het. D. (LOH),Amplification,Het. D. (LOH),Het. D. (LOH),Het. D. (LOH),Neutral,Neutral,Neutral,Neutral
3,3,13T02-13N02,13,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,...,0.56,Polyploid LOH (Absolute Gain),Focal Gain,Neutral,Post-WGD Loss,Diploid and LOH after Post-WGD-Loss,Amplification,Diploid and LOH after Post-WGD-Loss,Polyploid LOH (Absolute Gain),Deep Deletion
4,4,17_439_00067_T2-17_439_00067_WB,17_439_00067,DFCI_with_snRNAseq,Female,NaN,10.0,"Recurrent, Metastatic",NaN,No Metastasis,...,0.83,Diploid and LOH after Post-WGD-Loss,Focal Gain,Diploid after Post-WGD-loss,Diploid after Post-WGD-loss,Severe Post-WGD loss (LOH),Neutral,Focal Gain,Diploid and LOH after Post-WGD-Loss,Diploid and LOH after Post-WGD-Loss
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,231,WGS_12_15-WGS_11_13,participant_21,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,...,0.90,Het. D. (LOH),Not_available,Neutral,Neutral,Het. D. (LOH),Neutral,Neutral,Neutral,Neutral
232,232,WGS_12_19-WGS_11_17,participant_47,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,...,0.76,Polyploid LOH (Absolute Gain),Neutral,Neutral,Neutral,Severe Post-WGD loss (LOH),Focal Gain,Diploid after Post-WGD-loss,Diploid after Post-WGD-loss,Amplification
233,233,WGS_12_31-WGS_11_29,participant_31,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,...,0.30,Absolute Gain/Relative Loss,Neutral,Not_available,Not_available,Diploid and LOH after Post-WGD-Loss,Neutral,Neutral,Neutral,Diploid and LOH after Post-WGD-Loss
234,235,WGS_12_7-WGS_11_5,participant_37,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,...,0.73,Not_available,Amplification,Neutral,Polyploid LOH (Absolute Gain),Post-WGD Loss (LOH),Focal Gain,Post-WGD Loss (LOH),Absolute Gain/Relative Loss,Post-WGD Loss


### Absolute Total Copy number and LOH annotation

In [ ]:
# Using ABSOLUTE segment file output to annotate TCN and LOH status in each sample for target genes. 
# So this file you should get from ABSOLUTE, LOH indicates if for a segment loss of heterozygosity (LOH) occured or not.
#  Note that for X chromosome no copy number data is available and therefore ATRX is all NaN. 
cn_loh = pd.read_csv('../../../CNVs_OS/Absolute/2026_07_17_OS_data_segment_file_TCN_LOH_for_tumor_suppressors.tsv', sep='\t')

In [74]:
cn_loh

,Unnamed: 0,sample,gene,Chromosome,gene_start,gene_end,total_cn,minor_cn,LOH,n_segments_overlapping,CN_Status_Detail,LOH_final
0,0,09T02-09N01,CDKN2A,9,21967752,21995301,4.0,2.0,False,1,CN_available,False
1,1,09T03-09N01,CDKN2A,9,21967752,21995301,4.0,2.0,False,1,CN_available,False
2,2,10T03-10N02,CDKN2A,9,21967752,21995301,2.0,1.0,False,1,CN_available,False
3,3,13T02-13N02,CDKN2A,9,21967752,21995301,0.0,0.0,True,1,CN_available,True
4,4,17_439_00067_T2-17_439_00067_WB,CDKN2A,9,21967752,21995301,2.0,0.0,True,1,CN_available,True
...,...,...,...,...,...,...,...,...,...,...,...,...
1495,1495,WGS_12_19-WGS_11_17,ATRX,X,77504880,77786233,NaN,NaN,NaN,0,no_CN_caller_support_sex_chrom,NaN
1496,1496,WGS_12_31-WGS_11_29,ATRX,X,77504880,77786233,NaN,NaN,NaN,0,no_CN_caller_support_sex_chrom,NaN
1497,1497,WGS_12_38-WGS_11_32,ATRX,X,77504880,77786233,NaN,NaN,NaN,0,no_CN_caller_support_sex_chrom,NaN
1498,1498,WGS_12_7-WGS_11_5,ATRX,X,77504880,77786233,NaN,NaN,NaN,0,no_CN_caller_support_sex_chrom,NaN


In [75]:
# Pivot cn_loh from long to wide: one row per sample, columns like TP53_total_cn, TP53_LOH, etc.
cn_loh_wide = cn_loh.pivot(index='sample', columns='gene', values=['total_cn', 'LOH_final'])

# Flatten the multi-level columns: ('total_cn', 'TP53') -> 'TP53_total_cn'
cn_loh_wide.columns = [f"{gene}_{measure}" for measure, gene in cn_loh_wide.columns]
cn_loh_wide = cn_loh_wide.reset_index()
cn_loh_wide

,sample,ATRX_total_cn,CDKN2A_total_cn,NF2_total_cn,PTEN_total_cn,RB1_total_cn,TP53_total_cn,ATRX_LOH_final,CDKN2A_LOH_final,NF2_LOH_final,PTEN_LOH_final,RB1_LOH_final,TP53_LOH_final
0,09T02-09N01,NaN,4.0,2.0,2.0,2.0,2.0,NaN,False,True,False,True,True
1,09T03-09N01,NaN,4.0,2.0,2.0,2.0,2.0,NaN,False,True,False,True,True
2,10T03-10N02,NaN,2.0,1.0,2.0,1.0,1.0,NaN,False,True,False,True,True
3,13T02-13N02,NaN,0.0,5.0,5.0,2.0,5.0,NaN,True,False,True,True,True
4,17_439_00067_T2-17_439_00067_WB,NaN,2.0,2.0,2.0,1.0,2.0,NaN,True,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,WGS_12_19-WGS_11_17,NaN,6.0,2.0,2.0,3.0,3.0,NaN,False,False,False,True,True
246,WGS_12_31-WGS_11_29,NaN,2.0,4.0,4.0,2.0,3.0,NaN,True,False,False,True,False
247,WGS_12_38-WGS_11_32,NaN,8.0,4.0,4.0,2.98364,3.0,NaN,False,False,False,True,True
248,WGS_12_7-WGS_11_5,NaN,3.0,4.0,4.0,3.0,NaN,NaN,False,False,False,True,True


In [76]:
# Rename every "<GENE>_LOH_final" column to "<GENE>_LOH"
cn_loh_wide = cn_loh_wide.rename(
    columns={col: col.replace('_LOH_final', '_LOH') for col in cn_loh_wide.columns if col.endswith('_LOH_final')}
)

# Merge into metadata
meta_new_annotated = meta_new_annotated.merge(
    cn_loh_wide,
    left_on='Tumor_Sample_Barcode',
    right_on='sample',
    how='left'
)

# Clean up
meta_new_annotated = meta_new_annotated.drop(columns=['sample'], errors='ignore')
if 'Unnamed: 0' in meta_new_annotated.columns:
    meta_new_annotated = meta_new_annotated.drop(columns=['Unnamed: 0'])

meta_new_annotated

,Tumor_Sample_Barcode,participant,study,sex,Ethnicity,Age_at_diagnosis,Sample_type,Location_Primary_Site,Metastatis_at_diagnosis,Metastasis_Status,...,NF2_total_cn,PTEN_total_cn,RB1_total_cn,TP53_total_cn,ATRX_LOH,CDKN2A_LOH,NF2_LOH,PTEN_LOH,RB1_LOH,TP53_LOH
0,09T02-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,2.0,2.0,2.0,2.0,NaN,False,True,False,True,True
1,09T03-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,2.0,2.0,2.0,2.0,NaN,False,True,False,True,True
2,10T03-10N02,10,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,1.0,2.0,1.0,1.0,NaN,False,True,False,True,True
3,13T02-13N02,13,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,5.0,5.0,2.0,5.0,NaN,True,False,True,True,True
4,17_439_00067_T2-17_439_00067_WB,17_439_00067,DFCI_with_snRNAseq,Female,NaN,10.0,"Recurrent, Metastatic",NaN,No Metastasis,Metastasis,...,2.0,2.0,1.0,2.0,NaN,True,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,WGS_12_15-WGS_11_13,participant_21,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,2.0,1.0,1.0,NaN,False,False,False,True,True
232,WGS_12_19-WGS_11_17,participant_47,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,2.0,3.0,3.0,NaN,False,False,False,True,True
233,WGS_12_31-WGS_11_29,participant_31,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.0,4.0,2.0,3.0,NaN,True,False,False,True,False
234,WGS_12_7-WGS_11_5,participant_37,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.0,4.0,3.0,NaN,NaN,False,False,False,True,True


# Mutation Status for TSGs



In [ ]:
# Load MAF file after removal of artifact mutations (signatureanalyzer output) 
maf = pd.read_csv('../../../MutSig_Driver_Analysis/mutsig_tool_files_MutSig2CV_standalone_macOS/SNVs_filtered_signatureanalyser/2026_02_27_GenomeNexus_output_enriched.maf', sep='\t')
maf


/var/folders/54/_6vs7vnn4433pfg0vbsbtjgw0000gn/T/ipykernel_56694/888629741.py:2: DtypeWarning: Columns (0: oncokb_highestSensitiveLevel) have mixed types. Specify dtype option on import or set low_memory=False.
  maf = pd.read_csv('../../../MutSig_Driver_Analysis/mutsig_tool_files_MutSig2CV_standalone_macOS/SNVs_filtered_signatureanalyser/2026_02_27_GenomeNexus_output_enriched.maf', sep='\t')


,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Consequence,Variant_Classification,...,oncokb_geneExist,oncokb_highestDXLevel,oncokb_highestPXLevel,oncokb_highestResistanceLevel,oncokb_highestSensitiveLevel,oncokb_mutationEffect,oncokb_mutationEffectCitations,oncokb_oncogenic,oncokb_variantExist,Annotation_Status
0,MTOR,2475.0,NaN,GRCh38,1,11114411,11114412,+,frameshift_variant,Frame_Shift_Ins,...,True,NaN,NaN,NaN,NaN,Unknown,NaN,Unknown,False,SUCCESS
1,AADACL3,126767.0,NaN,GRCh38,1,12727616,12727616,+,3_prime_UTR_variant,3'UTR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SUCCESS
2,IGSF21,84966.0,NaN,GRCh38,1,18376374,18376374,+,missense_variant,Missense_Mutation,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SUCCESS
3,CATSPER4,378807.0,NaN,GRCh38,1,26202660,26202660,+,3_prime_UTR_variant,3'UTR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SUCCESS
4,ZDHHC18,84243.0,NaN,GRCh38,1,26855096,26855096,+,3_prime_UTR_variant,3'UTR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SUCCESS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24853,LY6K,54742.0,NaN,GRCh38,8,142700625,142700625,+,missense_variant,Missense_Mutation,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SUCCESS
24854,TMEM215,401498.0,NaN,GRCh38,9,32787577,32787577,+,3_prime_UTR_variant,3'UTR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SUCCESS
24855,TUBB4B,10383.0,NaN,GRCh38,9,137242974,137242974,+,synonymous_variant,Silent,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SUCCESS
24856,CNKSR2,22866.0,NaN,GRCh38,X,21595036,21595036,+,synonymous_variant,Silent,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SUCCESS


In [78]:
# Define Target genes
target_genes = ['TP53', 'RB1', 'ATRX', 'CDKN2A', 'PTEN', 'NF2']
# Filter maf to only annotate target genes
filtered_maf = maf[maf['Hugo_Symbol'].isin(target_genes)]
mutation_status_df = filtered_maf[['Tumor_Sample_Barcode', 'Hugo_Symbol', "Variant_Classification", 'oncokb_oncogenic']]

In [79]:
# Group by Sample and Gene, then join the variants into one string
pivot_str = mutation_status_df.groupby(['Tumor_Sample_Barcode', 'Hugo_Symbol'])['Variant_Classification'] \
    .apply(lambda x: ', '.join(x.astype(str))) \
    .unstack()
pivot_str

Hugo_Symbol,ATRX,CDKN2A,NF2,PTEN,RB1,TP53
Tumor_Sample_Barcode,,,,,,
13T02-13N02,3'UTR,NaN,NaN,NaN,NaN,NaN
24T02-24N02,NaN,NaN,NaN,NaN,NaN,Missense_Mutation
41T01-41N03-MT-union,"3'UTR, Missense_Mutation, Missense_Mutation",NaN,NaN,NaN,NaN,NaN
CCG1106_005_T1-CCG1106_005_WB,NaN,NaN,NaN,NaN,Nonsense_Mutation,NaN
SA541864-SA541865,NaN,NaN,NaN,NaN,5'UTR,NaN
...,...,...,...,...,...,...
TARGET-40-PATUXZ-01A-TARGET-40-PATUXZ-10A,Missense_Mutation,NaN,NaN,NaN,NaN,NaN
WGS_12_11-WGS_11_9,NaN,NaN,NaN,NaN,Frame_Shift_Ins,NaN
WGS_12_19-WGS_11_17,NaN,NaN,NaN,NaN,Silent,NaN


In [80]:
# Rename columns for uniqueness later
pivot_str.rename(columns={gene : f'{gene}_Mutation_Status' for gene in target_genes}, inplace=True)
pivot_str.reset_index(inplace=True)
# Remove the axis name specifically
pivot_str = pivot_str.rename_axis(None, axis=1)

# Drop the column named 'index' if it was created and you don't need it
if 'index' in pivot_str.columns:
    pivot_str.drop(columns=['index'], inplace=True)

In [81]:
# Merge the mutataions to metadata (meta_new_annotated from above)
metadata_anno = pd.merge(meta_new_annotated, pivot_str, left_on='Union_Sample_Name', right_on='Tumor_Sample_Barcode', how='left')
metadata_anno.drop(columns=['Tumor_Sample_Barcode_y'], inplace=True)
metadata_anno.rename(columns={'Tumor_Sample_Barcode_x': 'Tumor_Sample_Barcode'}, inplace=True)
metadata_anno

,Tumor_Sample_Barcode,participant,study,sex,Ethnicity,Age_at_diagnosis,Sample_type,Location_Primary_Site,Metastatis_at_diagnosis,Metastasis_Status,...,NF2_LOH,PTEN_LOH,RB1_LOH,TP53_LOH,ATRX_Mutation_Status,CDKN2A_Mutation_Status,NF2_Mutation_Status,PTEN_Mutation_Status,RB1_Mutation_Status,TP53_Mutation_Status
0,09T02-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,True,False,True,True,NaN,NaN,NaN,NaN,NaN,NaN
1,09T03-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,True,False,True,True,NaN,NaN,NaN,NaN,NaN,NaN
2,10T03-10N02,10,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,True,False,True,True,NaN,NaN,NaN,NaN,NaN,NaN
3,13T02-13N02,13,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,False,True,True,True,3'UTR,NaN,NaN,NaN,NaN,NaN
4,17_439_00067_T2-17_439_00067_WB,17_439_00067,DFCI_with_snRNAseq,Female,NaN,10.0,"Recurrent, Metastatic",NaN,No Metastasis,Metastasis,...,False,True,True,True,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,WGS_12_15-WGS_11_13,participant_21,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,True,True,NaN,NaN,NaN,NaN,NaN,NaN
232,WGS_12_19-WGS_11_17,participant_47,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,True,True,NaN,NaN,NaN,NaN,Silent,NaN
233,WGS_12_31-WGS_11_29,participant_31,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,True,False,NaN,NaN,NaN,NaN,NaN,NaN
234,WGS_12_7-WGS_11_5,participant_37,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,True,True,NaN,NaN,NaN,NaN,NaN,NaN


In [82]:
# Add a column to define if a gene has a mutation or not (if multiple as summary) and add columns to split into 1 and 2. 
# Final cleanup
metadata_anno = metadata_anno.replace(['None', 'nan', ''], np.nan)

In [ ]:
# Fill NAs with 'No Mutation'
cols_to_fill = metadata_anno.filter(regex='Mutation_(Status|1|2)').columns
metadata_anno[cols_to_fill] = metadata_anno[cols_to_fill].fillna('No Mutation')
# Control if it worked
metadata_anno[['Tumor_Sample_Barcode','NF2_Mutation_Status', 'PTEN_Mutation_Status', 'RB1_Mutation_Status', 'TP53_Mutation_Status' ]]

In [ ]:
# 1. Join all OncoKB statuses into one string per Gene/Sample
# Use .dropna() to avoid "nan, nan" strings
pivot_oncokb = mutation_status_df.groupby(['Tumor_Sample_Barcode', 'Hugo_Symbol'])['oncokb_oncogenic'] \
    .apply(lambda x: ', '.join(x.dropna().astype(str))) \
    .unstack()

# 2. Reset index to make 'Tumor_Sample_Barcode' a column for easy merging
pivot_oncokb = pivot_oncokb.reset_index().rename_axis(None, axis=1)
pivot_oncokb

,Tumor_Sample_Barcode,ATRX,CDKN2A,NF2,PTEN,RB1,TP53
0,13T02-13N02,,NaN,NaN,NaN,NaN,NaN
1,24T02-24N02,NaN,NaN,NaN,NaN,NaN,Oncogenic
2,41T01-41N03-MT-union,"Unknown, Unknown",NaN,NaN,NaN,NaN,NaN
3,CCG1106_005_T1-CCG1106_005_WB,NaN,NaN,NaN,NaN,Likely Oncogenic,NaN
4,SA541864-SA541865,NaN,NaN,NaN,NaN,Unknown,NaN
...,...,...,...,...,...,...,...
75,TARGET-40-PATUXZ-01A-TARGET-40-PATUXZ-10A,Unknown,NaN,NaN,NaN,NaN,NaN
76,WGS_12_11-WGS_11_9,NaN,NaN,NaN,NaN,Likely Oncogenic,NaN
77,WGS_12_19-WGS_11_17,NaN,NaN,NaN,NaN,Unknown,NaN
78,WGS_12_38-WGS_11_32,Likely Oncogenic,NaN,NaN,NaN,Likely Oncogenic,NaN


In [ ]:
for gene in target_genes:
    if gene in pivot_oncokb.columns:
        # Create the new column names
        onc1 = f"{gene}_OncoKB_1"
        onc2 = f"{gene}_OncoKB_2"
        
        # Apply same handle_merges defined earlier
        pivot_oncokb[[onc1, onc2]] = pivot_oncokb[gene].apply(handle_merges)
        
        # Rename the original gene column to avoid confusion
        pivot_oncokb = pivot_oncokb.rename(columns={gene: f"{gene}_OncoKB_Status"})

# Now merge this into your main metadata_anno
metadata_anno = metadata_anno.merge(pivot_oncokb, left_on='Union_Sample_Name', right_on='Tumor_Sample_Barcode', how='left')
metadata_anno.drop(columns=['Tumor_Sample_Barcode_y'], inplace=True)
metadata_anno.rename(columns={'Tumor_Sample_Barcode_x': 'Tumor_Sample_Barcode'}, inplace=True)
metadata_anno

,Tumor_Sample_Barcode,participant,study,sex,Ethnicity,Age_at_diagnosis,Sample_type,Location_Primary_Site,Metastatis_at_diagnosis,Metastasis_Status,...,RB1_OncoKB_1,RB1_OncoKB_2,ATRX_OncoKB_1,ATRX_OncoKB_2,CDKN2A_OncoKB_1,CDKN2A_OncoKB_2,PTEN_OncoKB_1,PTEN_OncoKB_2,NF2_OncoKB_1,NF2_OncoKB_2
0,09T02-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,09T03-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10T03-10N02,10,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,13T02-13N02,13,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,17_439_00067_T2-17_439_00067_WB,17_439_00067,DFCI_with_snRNAseq,Female,NaN,10.0,"Recurrent, Metastatic",NaN,No Metastasis,Metastasis,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,WGS_12_15-WGS_11_13,participant_21,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
232,WGS_12_19-WGS_11_17,participant_47,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
233,WGS_12_31-WGS_11_29,participant_31,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
234,WGS_12_7-WGS_11_5,participant_37,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Fill NAs with 'No Mutations' and empty cells with 'No OncoKB Annotation'
cols_filler = metadata_anno.filter(regex='OncoKB_Status').columns
metadata_anno[cols_filler] = metadata_anno[cols_filler].fillna('No Mutation').replace('', 'No OncoKB Annotation')

# SV Annotation to metadata for oncoprint

Annotate consensus structural variants (SVs) with AnnotSV gene names and
flag which samples carry an SV in a predefined set of oncogenes /
tumor suppressor genes. The resulting per-sample gene columns are merged
into the sample metadata table.



In [92]:
# Load consensus SVs and keep only calls supported by both callers
sv_file = pd.read_hdf('../../../Structural_Variants_SVs/sv_file_dup_cleaned.h5')
#sv_file = sv_file[(sv_file['SvABA'] == 1) & (sv_file['Manta'] == 1)]


In [93]:
# Load and reshape the AnnotSV gene annotation

annotsv = pd.read_csv('../../../Structural_Variants_SVs/sv_calls_v2.annotated.tsv', sep='\t')

# Non-BND SVs (DEL/DUP/INV/INS): use the "full" annotation rows only, since split rows duplicate the same gene list per SV.
annotsv_simple = annotsv[(annotsv['SV_type'] != 'BND') & (annotsv['Annotation_mode'] == 'full')]

annotsv_def = annotsv_simple.rename(columns={
    'SV_chrom': 'chr1',
    'SV_start': 'pos1',
    'SV_end': 'pos2',
}).copy()

# Standardize chromosome naming (X/Y -> 23/24) to match the consensus SV file
annotsv_def['chr1'] = annotsv_def['chr1'].astype(str).replace({"X": "23", "Y": "24"})
annotsv_def['chr1'] = pd.to_numeric(annotsv_def['chr1']).astype(int)
annotsv_def['chr2'] = annotsv_def['chr1']

# AnnotSV coordinates are 1-based; shift pos1 to match the 0-based consensus file
annotsv_def['pos1'] = annotsv_def['pos1'] - 1

annotsv_def.rename(columns={'Gene_name': 'Collapsed_Gene_Name'}, inplace=True)
annotsv_to_merge1 = annotsv_def[['Samples_ID', 'chr1', 'chr2', 'pos1', 'pos2', 'id', 'Collapsed_Gene_Name']]


/var/folders/54/_6vs7vnn4433pfg0vbsbtjgw0000gn/T/ipykernel_56694/3364428416.py:3: DtypeWarning: Columns (0: SV_chrom, 1: P_gain_phen, 2: P_gain_hpo, 3: P_gain_source, 4: P_gain_coord, 5: P_loss_hpo, 6: P_ins_phen, 7: P_ins_hpo, 8: P_ins_source, 9: P_ins_coord, 10: po_P_gain_phen, 11: po_P_gain_source, 12: po_P_gain_coord, 13: po_P_gain_percent, 14: po_P_loss_phen, 15: po_P_loss_source, 16: po_P_loss_coord, 17: po_P_loss_percent, 18: B_ins_source, 19: B_ins_coord, 20: GC_content_left, 21: GC_content_right, 22: Gap_left, 23: Gap_right, 24: ENCODE_blacklist_left, 25: ENCODE_blacklist_characteristics_left, 26: ENCODE_blacklist_right, 27: ENCODE_blacklist_characteristics_right, 28: ACMG, 29: AnnotSV_ranking_criteria) have mixed types. Specify dtype option on import or set low_memory=False.
  annotsv = pd.read_csv('../../../Structural_Variants_SVs/sv_calls_v2.annotated.tsv', sep='\t')


In [94]:

# BND SVs: each breakend is reported as two separate rows sharing an 'id'. Collapse the gene names per id, then pair the two rows back
# together so each SV has a single chr1/pos1/chr2/pos2 record. ---
annotsv_bnd = annotsv[(annotsv['SV_type'] == 'BND') & (annotsv['Annotation_mode'] == 'full')]
annotsv_bnd['Gene_name'] = annotsv_bnd['Gene_name'].replace(np.nan, "NaN")

annotsv_transfbnd = (
    annotsv_bnd.groupby('id')['Gene_name']
    .apply(lambda x: ";".join(sorted(set(x.astype(str)))))
    .reset_index()
    .rename(columns={'Gene_name': 'Collapsed_Gene_Name'})
)
annotsv_bnd2 = annotsv_bnd.merge(annotsv_transfbnd, how='left', on='id')

annotsv_bnd3 = annotsv_bnd2.rename(columns={
    'SV_chrom': 'chr1',
    'SV_start': 'pos1',
    'SV_end': 'pos2',
}).copy()
annotsv_bnd3['pos1'] = annotsv_bnd3['pos1'] - 1
annotsv_bnd3['pos2'] = annotsv_bnd3['pos2'] - 1
annotsv_bnd3['chr1'] = annotsv_bnd3['chr1'].astype(str).replace({"X": "23", "Y": "24"})
annotsv_bnd3['chr1'] = pd.to_numeric(annotsv_bnd3['chr1']).astype(int)



In [95]:
annotsv_bnd4 = split_BNDs_position_fast(annotsv_bnd3)
annotsv_bnd5 = (
    annotsv_bnd4[['Samples_ID', 'chr1', 'chr2', 'pos1', 'pos2', 'id', 'Collapsed_Gene_Name']]
    .sort_values('id')
)

# Combine non-BND and BND gene annotation into one lookup table
intersection_annotation = pd.concat([annotsv_to_merge1, annotsv_bnd5], ignore_index=True)



In [ ]:
# Here one can use the intersection of SV callers if uncommented. In this paper we use the union.
sv_interc = sv_file#[(sv_file['SvABA']== 1) & (sv_file['Manta']== 1)]
sv_intersect = sv_interc.merge(intersection_annotation, how='left', left_on=['name','chr1','chr2', 'pos1', 'pos2'], right_on=['Samples_ID', 'chr1', 'chr2', 'pos1', 'pos2'])

In [97]:
# Merge gene annotation onto the consensus SVs
sv_intersect = sv_interc.merge(
    intersection_annotation,
    how='left',
    left_on=['name', 'chr1', 'chr2', 'pos1', 'pos2'],
    right_on=['Samples_ID', 'chr1', 'chr2', 'pos1', 'pos2'],
)

# Remove duplicate breakpoint entries (keep first), but save what was removed
dup_subset = ['name', 'chr1', 'pos1', 'chr2', 'pos2']
removed_duplicates = sv_intersect[sv_intersect.duplicated(subset=dup_subset, keep='first')]
sv_intersection_final = sv_intersect.drop_duplicates(subset=dup_subset, keep='first', ignore_index=True)

# Restrict to the final sample cohort
final_samples = metadata_anno['Tumor_Sample_Barcode'].unique()
sv_intersection_final = sv_intersection_final[sv_intersection_final['Samples_ID'].isin(final_samples)]

In [ ]:
# Download a GTF file downloaded (e.g. from GENCODE Basic Gene Annotation) and provide Path to the gtf file. 
#  ---- 1. Parse GTF for target gene coordinates ----
gtf_path = "/Users/thorsten/Postdoc/Reference Files/gencode.v49.annotation.gtf"
# Define target genes again
target_genes = ['TP53', 'MYC', 'CDK4', 'RB1', 'CCND3', 'CCNE1', 'PTEN', 'ATRX', 'CDKN2A', 'NF2']

# Read GTF, filter it for target genes a
gtf_cols = ['chrom', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']
gtf = pd.read_csv(gtf_path, sep='\t', comment='#', names=gtf_cols, dtype=str)

gtf_genes = gtf[gtf['feature'] == 'gene'].copy()
gtf_genes['gene_name'] = gtf_genes['attribute'].str.extract(r'gene_name "([^"]+)"')
gtf_genes = gtf_genes[gtf_genes['gene_name'].isin(target_genes)][['chrom', 'start', 'end', 'gene_name']]
# Standardize Chromosome format (X = 23, Y=24, no 'chr' in front of chromsome number, Start and end are integers)
gtf_genes['chrom'] = gtf_genes['chrom'].str.replace('chr', '', regex=False)
gtf_genes['chrom'] = gtf_genes['chrom'].replace({'X': '23', 'Y': '24'})
gtf_genes = gtf_genes[gtf_genes['chrom'].str.isnumeric()]
gtf_genes['chrom'] = gtf_genes['chrom'].astype(int)
gtf_genes['start'] = gtf_genes['start'].astype(int)
gtf_genes['end']   = gtf_genes['end'].astype(int)

# Define leftmost start and rightmost end for each gene
gene_bounds = (
    gtf_genes.groupby('gene_name')
    .agg(chrom=('chrom', 'first'), start=('start', 'min'), end=('end', 'max'))
    .reset_index()
)

# ---- 2. Load the SV signature file to get reliable svclass labels ----
sig_bedpe = pd.read_csv('../../../Structural_Variants_SVs/sv_signature_analysis.bedpe', sep='\t',
    dtype={'chrom1': str, 'chrom2': str}
)

# Standardize chrom naming to match sv_intersection_final (X/Y -> 23/24, numeric)
sig_bedpe['chrom1'] = sig_bedpe['chrom1'].replace({'X': '23', 'Y': '24'})
sig_bedpe['chrom2'] = sig_bedpe['chrom2'].replace({'X': '23', 'Y': '24'})
sig_bedpe = sig_bedpe[sig_bedpe['chrom1'].str.isnumeric() & sig_bedpe['chrom2'].str.isnumeric()]
sig_bedpe['chrom1'] = sig_bedpe['chrom1'].astype(int)
sig_bedpe['chrom2'] = sig_bedpe['chrom2'].astype(int)

# Merge svclass onto sv_intersection_final by sample + breakpoint coordinates
# Adjust merge keys if your bedpe start/end differ by an offset (e.g. 0- vs 1-based)
# from pos1/pos2 in sv_intersection_final -- check a few known rows to confirm.
sv_intersection_final = sv_intersection_final.merge(
    sig_bedpe[['sample', 'chrom1', 'end1', 'chrom2', 'end2', 'svclass']],
    how='left',
    left_on=['Samples_ID', 'chr1', 'pos1', 'chr2', 'pos2'],
    right_on=['sample', 'chrom1', 'end1', 'chrom2', 'end2'],
)

# ---- 3. Flag SVs per gene ----
# Default rule (all genes): breakpoint within gene body ONLY
# Special rule for ATRX: also include spans-gene-body AND svclass == 'deletion'

for _, row in gene_bounds.iterrows():
    gene, chrom, g_start, g_end = row['gene_name'], row['chrom'], row['start'], row['end']

    same_chrom = (sv_intersection_final['chr1'] == chrom) & (sv_intersection_final['chr2'] == chrom)

    # Condition 1: breakpoint within gene body (applies to ALL genes)
    bp1_in_gene = (sv_intersection_final['chr1'] == chrom) & \
                  (sv_intersection_final['pos1'].between(g_start, g_end))
    bp2_in_gene = (sv_intersection_final['chr2'] == chrom) & \
                  (sv_intersection_final['pos2'].between(g_start, g_end))
    breakpoint_hit = bp1_in_gene | bp2_in_gene

    col_name = f"{gene}_SV_gene_body"

    if gene == "ATRX":
        # Condition 2 (ATRX ONLY): SV fully spans the gene AND is a deletion
        spans_gene = same_chrom & \
                     (sv_intersection_final['pos1'] <= g_start) & \
                     (sv_intersection_final['pos2'] >= g_end)
        is_deletion = (sv_intersection_final['svclass'] == 'deletion') | \
              (sv_intersection_final['class'] == 'deletion')
        span_deletion_hit = spans_gene & is_deletion

        sv_intersection_final[col_name] = np.where(
            breakpoint_hit | span_deletion_hit, "SV", "No"
        )
    else:
        # All other genes: breakpoint-in-gene-body only, no spanning exception
        sv_intersection_final[col_name] = np.where(breakpoint_hit, "SV", "No")

In [ ]:
# Short control if ATRX SVs were recognized
sv_intersection_final['ATRX_SV_gene_body'].value_counts()

ATRX_SV_gene_body
No    65279
SV       93
Name: count, dtype: int64

In [ ]:
# Copy metadata_anno
metadata_anno_final = metadata_anno.copy()

# 4. Flag SVs in target oncogenes / tumor suppressor genes and annotate metadata
rename_class = {gene: f"{gene}_SV_all_class" for gene in target_genes}
rename_id = {gene: f"{gene}_SV_all_id" for gene in target_genes}

# Keep only SVs whose (collapsed) gene annotation includes a target gene
pattern = r'\b(?:' + '|'.join(target_genes) + r')\b'
df_targets = sv_intersection_final[
    sv_intersection_final['Collapsed_Gene_Name'].str.contains(pattern, na=False, regex=True)
]

# One SV can list several genes in 'Collapsed_Gene_Name' (';'-separated) -> one row per gene
df_exploded = df_targets.copy()
df_exploded['Collapsed_Gene_Name'] = df_exploded['Collapsed_Gene_Name'].str.split(';')
df_exploded = df_exploded.explode('Collapsed_Gene_Name')
df_exploded['Collapsed_Gene_Name'] = df_exploded['Collapsed_Gene_Name'].replace('NaN', np.nan)
df_exploded.dropna(subset=['Collapsed_Gene_Name'], inplace=True)

# Aggregate SV class and SV id per sample/gene
df_sv_grouped = (
    df_exploded.groupby(['Samples_ID', 'Collapsed_Gene_Name'])
    .agg({
        'class': lambda x: ';'.join(map(str, x)),
        'id': lambda x: ';'.join(map(str, sorted(set(x)))),
        'pos1': 'count',
    })
    .reset_index()
    .rename(columns={'pos1': 'Number_of_SVs_in_Gene', 'class': 'SV_class', 'Collapsed_Gene_Name': 'Hugo_Symbol'})
)
df_to_pivot = df_sv_grouped[df_sv_grouped['Hugo_Symbol'].isin(target_genes)]

# Wide format: one column per gene, one row per sample
pivot_svs_class = (
    df_to_pivot.pivot(columns='Hugo_Symbol', index='Samples_ID', values='SV_class')
    .reset_index()
    .rename_axis(None, axis=1)
    .rename(columns=rename_class)
)
pivot_svs_id = (
    df_to_pivot.pivot(columns='Hugo_Symbol', index='Samples_ID', values='id')
    .reset_index()
    .rename_axis(None, axis=1)
    .rename(columns=rename_id)
)

# ---- 5. Aggregate gene-body SV flags to one row per sample and merge in ----

gene_body_cols = [f"{gene}_SV_gene_body" for gene in target_genes if f"{gene}_SV_gene_body" in sv_intersection_final.columns]

# For each gene-body column: a sample is "SV" if ANY of its SV rows say "SV"
gene_body_per_sample = (
    sv_intersection_final
    .groupby('Samples_ID')[gene_body_cols]
    .apply(lambda x: (x == 'SV').any())
    .reset_index()
)

# Convert True/False -> "SV"/"No" to match your existing convention
for col in gene_body_cols:
    gene_body_per_sample[col] = np.where(gene_body_per_sample[col], "SV", "No")

# Merge into metadata_anno_final
metadata_anno_final = metadata_anno_final.merge(
    gene_body_per_sample,
    how='left',
    left_on='Tumor_Sample_Barcode',
    right_on='Samples_ID'
).drop(columns=['Samples_ID'])

# Fill NA (samples with zero SVs anywhere) as "No"
metadata_anno_final[gene_body_cols] = metadata_anno_final[gene_body_cols].fillna('No')


In [103]:
# add a column to remove multi-tumor samples if necessary
filter_list = pd.read_csv( "../../../Sample_list_for_filtering.tsv", sep="\t")

# Keep only the join key + filter status
filter_status_only = filter_list[["tumor_normal_pair", "filter status"]]

# Left join: keep all rows in metadata_anno_final, attach filter status where matched
metadata_anno_final = metadata_anno_final.merge(
    filter_status_only,
    left_on="Tumor_Sample_Barcode",
    right_on="tumor_normal_pair",
    how="left"
)

# Drop the redundant join-key column from the right table (pandas keeps both by default)
metadata_anno_final = metadata_anno_final.drop(columns=["tumor_normal_pair"])

metadata_anno_final

/var/folders/54/_6vs7vnn4433pfg0vbsbtjgw0000gn/T/ipykernel_56694/3334459664.py:2: DtypeWarning: Columns (0: tumor_normal_pair, 1: cohort, 2: hapaseg file status, 3: absolute CN calls file status, 4: SV/SNV calls file status, 5: filter status, 6: notes) have mixed types. Specify dtype option on import or set low_memory=False.
  filter_list = pd.read_csv( "../../../Sample_list_for_filtering.tsv", sep="\t")


,Tumor_Sample_Barcode,participant,study,sex,Ethnicity,Age_at_diagnosis,Sample_type,Location_Primary_Site,Metastatis_at_diagnosis,Metastasis_Status,...,MYC_SV_gene_body,CDK4_SV_gene_body,RB1_SV_gene_body,CCND3_SV_gene_body,CCNE1_SV_gene_body,PTEN_SV_gene_body,ATRX_SV_gene_body,CDKN2A_SV_gene_body,NF2_SV_gene_body,filter status
0,09T02-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,No,No,SV,No,SV,No,No,No,No,NaN
1,09T03-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,No,No,SV,No,No,No,No,No,No,multiple-tumor
2,10T03-10N02,10,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,No,No,No,No,No,No,No,No,No,NaN
3,13T02-13N02,13,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,No,No,No,No,No,No,No,No,No,NaN
4,17_439_00067_T2-17_439_00067_WB,17_439_00067,DFCI_with_snRNAseq,Female,NaN,10.0,"Recurrent, Metastatic",NaN,No Metastasis,Metastasis,...,No,No,No,No,No,No,No,No,No,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,WGS_12_15-WGS_11_13,participant_21,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,No,No,No,No,No,No,No,No,NaN
232,WGS_12_19-WGS_11_17,participant_47,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,No,SV,SV,No,No,No,No,No,NaN
233,WGS_12_31-WGS_11_29,participant_31,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,No,No,No,No,No,SV,No,No,NaN
234,WGS_12_7-WGS_11_5,participant_37,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,No,No,No,No,No,No,No,No,NaN


In [ ]:
#Clean up columns
metadata_anno_final = metadata_anno_final.drop(columns=['sample_x', 'sample_y'])
#Create normal and Tumor_ID columns (required later)
metadata_anno_final['Tumor_ID'] = metadata_anno_final['Tumor_Sample_Barcode'].apply(extract_tumor_id)
metadata_anno_final['Normal_ID'] = metadata_anno_final['Tumor_Sample_Barcode'].apply(extract_normal_id)

# Add ecDNA Status of sample

In [106]:
ecDNA = pd.read_csv('../../../ecDNA/amplicon_architect_aa_result_table_consensus.tsv', sep='\t')
ecDNA

,Sample name,AA amplicon number,Feature ID,Classification,Location,Oncogenes,All genes,NCBI Gene IDs,Complexity score,ecDNA context,...,Feature BED file,CNV BED file,AS-p version,AA version,AC version,AA PNG file,AA PDF file,AA summary file,Run metadata JSON,Sample metadata JSON
0,09T02,13.0,09T02_amplicon13_Linear_1,Linear,['chr9:634000-815576'],[],['KANK1'],['NM_001256876'],0.691437,NaN,...,/mnt/disks/cromwell_root/09T02_classification/...,Not provided,1.5.1,1.5.r5,1.5.1,/mnt/disks/cromwell_root/09T02_AA_results/09T0...,/mnt/disks/cromwell_root/09T02_AA_results/09T0...,/mnt/disks/cromwell_root/09T02_AA_results/09T0...,/mnt/disks/cromwell_root/09T02_run_metadata.json,/mnt/disks/cromwell_root/09T02_sample_metadata...
1,09T02,3.0,09T02_amplicon3_ecDNA_1,ecDNA,['chr19:12293891-12440217'],[],"['ZNF44', 'ZNF442', 'ZNF443', 'ZNF563', 'ZNF799']","['NM_001353551', 'NM_030824', 'NM_005815', 'NM...",0.565933,Unknown,...,/mnt/disks/cromwell_root/09T02_classification/...,Not provided,1.5.1,1.5.r5,1.5.1,/mnt/disks/cromwell_root/09T02_AA_results/09T0...,/mnt/disks/cromwell_root/09T02_AA_results/09T0...,/mnt/disks/cromwell_root/09T02_AA_results/09T0...,/mnt/disks/cromwell_root/09T02_run_metadata.json,/mnt/disks/cromwell_root/09T02_sample_metadata...
2,09T02,3.0,09T02_amplicon3_ecDNA_2,ecDNA,"['chr19:10110051-10110079', 'chr19:10428908-10...","['DNM2', 'SMARCA4']","['AP1M2', 'ATG4D', 'CDKN2D', 'DNM2', 'ILF3', '...","['NM_005498', 'NR_104025', 'NM_001800', 'NM_00...",0.998827,Heavily rearranged multichromosomal,...,/mnt/disks/cromwell_root/09T02_classification/...,Not provided,1.5.1,1.5.r5,1.5.1,/mnt/disks/cromwell_root/09T02_AA_results/09T0...,/mnt/disks/cromwell_root/09T02_AA_results/09T0...,/mnt/disks/cromwell_root/09T02_AA_results/09T0...,/mnt/disks/cromwell_root/09T02_run_metadata.json,/mnt/disks/cromwell_root/09T02_sample_metadata...
3,09T02,4.0,09T02_amplicon4_ecDNA_1,ecDNA,"['chr8:117647510-117648469', 'chr8:117648607-1...",['EXT1'],['EXT1'],['NM_000127'],0.512751,Unknown,...,/mnt/disks/cromwell_root/09T02_classification/...,Not provided,1.5.1,1.5.r5,1.5.1,/mnt/disks/cromwell_root/09T02_AA_results/09T0...,/mnt/disks/cromwell_root/09T02_AA_results/09T0...,/mnt/disks/cromwell_root/09T02_AA_results/09T0...,/mnt/disks/cromwell_root/09T02_run_metadata.json,/mnt/disks/cromwell_root/09T02_sample_metadata...
4,09T03,1.0,09T03_amplicon1_ecDNA_1,ecDNA,"['chr19:10455066-10840920', 'chr19:11465100-12...","['CALR', 'DNM2', 'JUNB', 'LYL1']","['ACP5', 'ADGRE2', 'ADGRE3', 'AP1M2', 'ATG4D',...","['NM_001611', 'NM_013447', 'NM_001289159', 'NM...",0.710648,Heavily rearranged multichromosomal,...,/mnt/disks/cromwell_root/09T03_classification/...,Not provided,1.5.1,1.5.r5,1.5.1,/mnt/disks/cromwell_root/09T03_AA_results/09T0...,/mnt/disks/cromwell_root/09T03_AA_results/09T0...,/mnt/disks/cromwell_root/09T03_AA_results/09T0...,/mnt/disks/cromwell_root/09T03_run_metadata.json,/mnt/disks/cromwell_root/09T03_sample_metadata...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,WGS_12_7,22.0,WGS_12_7_amplicon22_Complex-non-cyclic_1,Complex-non-cyclic,"['chr8:95182275-95184287', 'chr8:97246749-9727...","['CCN3', 'CCN4', 'MYC', 'NDRG1', 'PVT1', 'TRIB1']","['ADCY8', 'ANXA13', 'ASAP1', 'ASAP1-IT1', 'ASA...","['NM_001115', 'NM_004306', 'NM_001362924', 'NR...",2.019606,NaN,...,/mnt/disks/cromwell_root/WGS_12_7_classificati...,Not provided,1.5.1,1.5.r5,1.5.1,/mnt/disks/cromwell_root/WGS_12_7_AA_results/W...,/mnt/disks/cromwell_root/WGS_12_7_AA_results/W...,/mnt/disks/cromwell_root/WGS_12_7_AA_results/W...,/mnt/disks/cromwell_root/WGS_12_7_run_metadata...,/mnt/disks/cromwell_root/WGS_12_7_sample_metad...
887,WGS_12_7,2.0,WGS_12_7_amplicon2_ecDNA_1,ecDNA,"['chr9:1668589-1668887', 'chr9:1848205-1876009...",[],[],[],0.918403,Unknown,...,/mnt/disks/cromwell_root/WGS_12_7_classificati...,Not provided,1.5.1,1.5.r5,1.5.1,/mnt/disks/cromwell_root/WGS_12_7_AA_results/W...,/mnt/disks/cromwell_root/WGS_12_7_AA_results/W...,/mnt/disks/cromw

In [107]:
# Step 1: flag ecDNA per amplicon row
ecDNA['has_ecDNA_row'] = ecDNA['Classification'] == 'ecDNA'

# Step 2: aggregate to sample level - yes if ANY amplicon in that sample is ecDNA
ecDNA_sample_level = (
    ecDNA.groupby('Sample name')['has_ecDNA_row']
    .any()
    .reset_index()
)
ecDNA_sample_level['has_ecDNA'] = ecDNA_sample_level['has_ecDNA_row'].map({True: 'yes', False: 'no'})
ecDNA_sample_level = ecDNA_sample_level.drop(columns=['has_ecDNA_row'])

ecDNA_sample_level

,Sample name,has_ecDNA
0,09T02,yes
1,09T03,yes
2,10T03,yes
3,13T02,no
4,17_439_00067_T2,yes
...,...,...
187,WGS_12_15,no
188,WGS_12_19,no
189,WGS_12_31,yes
190,WGS_12_7,yes


In [ ]:
# Merge to the metadata table
metadata_anno_final = metadata_anno_final.merge(
    ecDNA_sample_level[['Sample name', 'has_ecDNA']],
    left_on='Tumor_ID',        # or whichever column holds the sample name in metadata_anno_final
    right_on='Sample name',
    how='left'
)

# fill missing (not in ecDNA table) with 'NA' string
metadata_anno_final['has_ecDNA'] = metadata_anno_final['has_ecDNA'].fillna('NA')

# drop the redundant join key
meta_ecDNA = metadata_anno_final.drop(columns=['Sample name'])
meta_ecDNA

,Tumor_Sample_Barcode,participant,study,sex,Ethnicity,Age_at_diagnosis,Sample_type,Location_Primary_Site,Metastatis_at_diagnosis,Metastasis_Status,...,CCND3_SV_gene_body,CCNE1_SV_gene_body,PTEN_SV_gene_body,ATRX_SV_gene_body,CDKN2A_SV_gene_body,NF2_SV_gene_body,filter status,Tumor_ID,Normal_ID,has_ecDNA
0,09T02-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,No,SV,No,No,No,No,NaN,09T02,09N01,yes
1,09T03-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,No,No,No,No,No,No,multiple-tumor,09T03,09N01,yes
2,10T03-10N02,10,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,No,No,No,No,No,No,NaN,10T03,10N02,yes
3,13T02-13N02,13,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,No,No,No,No,No,No,NaN,13T02,13N02,no
4,17_439_00067_T2-17_439_00067_WB,17_439_00067,DFCI_with_snRNAseq,Female,NaN,10.0,"Recurrent, Metastatic",NaN,No Metastasis,Metastasis,...,No,No,No,No,No,No,NaN,17_439_00067_T2,17_439_00067_WB,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,WGS_12_15-WGS_11_13,participant_21,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,No,No,No,No,No,NaN,WGS_12_15,WGS_11_13,no
232,WGS_12_19-WGS_11_17,participant_47,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,SV,No,No,No,No,No,NaN,WGS_12_19,WGS_11_17,no
233,WGS_12_31-WGS_11_29,participant_31,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,No,No,SV,No,No,NaN,WGS_12_31,WGS_11_29,yes
234,WGS_12_7-WGS_11_5,participant_37,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,No,No,No,No,No,NaN,WGS_12_7,WGS_11_5,yes


# Label Multi-tumor samples

In [111]:
# Run the function to extract the normal ID
meta_ecDNA['Normal_ID'] = meta_ecDNA['Tumor_Sample_Barcode'].apply(extract_normal_id)
# Identify patients with multiple samples by counting normal IDs
normal_counts = meta_ecDNA['Normal_ID'].value_counts()
# Get the indices (Normal ID names) from normal counts and save to a list
multi_tumor_normals = normal_counts[normal_counts > 1].index.tolist()

In [112]:
# Generate labels
gen = label_generator()
label_mapping = {normal: next(gen) for normal in multi_tumor_normals}
label_mapping

{'SJOS001101_G1': 'a',
 '41N03': 'b',
 'SJOS001107_G1': 'c',
 'SJOS030589_G2': 'd',
 '09N01': 'e',
 '39N01': 'f',
 'SJOS001105_G1': 'g',
 'SJOS001112_G1': 'h',
 'SJOS014_G': 'i',
 'SJOS010_G': 'j',
 'SJOS013768_G1': 'k',
 'SJOS030101_G1': 'l',
 'SJOS030422_G1': 'm',
 'SJOS030591_G1': 'n',
 'SJOS030605_G1': 'o',
 'SJOS030645_G1': 'p',
 'SJOS030759_G1': 'q',
 'SJOS046149_G2': 'r',
 'SJOS063833_G1': 's',
 'SJST033210_G1': 't',
 'SJST033229_G1': 'u'}

In [113]:
meta_ecDNA['Multi_Tumor_Label'] = meta_ecDNA['Normal_ID'].map(label_mapping).fillna('')

## Add number of SVs per sample 

In [114]:
# Count SVs per sample 
number_of_svs = sv_file.groupby('name')['name'].count().reset_index(name='number_of_SVs')
number_of_svs

,name,number_of_SVs
0,09T02-09N01,432
1,09T03-09N01,237
2,10T03-10N02,276
3,13T02-13N02,423
4,17_439_00067_T2-17_439_00067_WB,499
...,...,...
287,WGS_12_19-WGS_11_17,295
288,WGS_12_31-WGS_11_29,327
289,WGS_12_38-WGS_11_32,131
290,WGS_12_7-WGS_11_5,283


In [115]:
# Merge it to metadata
SV_number_added_meta = meta_ecDNA.merge(number_of_svs, left_on='Tumor_Sample_Barcode', right_on='name', how='left')
SV_number_added_meta = SV_number_added_meta.drop(columns='name')
SV_number_added_meta['number_of_SVs'] = SV_number_added_meta['number_of_SVs'].astype('Int64')
SV_number_added_meta

,Tumor_Sample_Barcode,participant,study,sex,Ethnicity,Age_at_diagnosis,Sample_type,Location_Primary_Site,Metastatis_at_diagnosis,Metastasis_Status,...,PTEN_SV_gene_body,ATRX_SV_gene_body,CDKN2A_SV_gene_body,NF2_SV_gene_body,filter status,Tumor_ID,Normal_ID,has_ecDNA,Multi_Tumor_Label,number_of_SVs
0,09T02-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,No,No,No,No,NaN,09T02,09N01,yes,e,432
1,09T03-09N01,9,DFCI_from_CZ_Zhang,NaN,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,No,No,No,No,multiple-tumor,09T03,09N01,yes,e,237
2,10T03-10N02,10,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,No,No,No,No,NaN,10T03,10N02,yes,,276
3,13T02-13N02,13,DFCI_from_CZ_Zhang,Male,NaN,NaN,Metastatic,NaN,NaN,Metastasis,...,No,No,No,No,NaN,13T02,13N02,no,,423
4,17_439_00067_T2-17_439_00067_WB,17_439_00067,DFCI_with_snRNAseq,Female,NaN,10.0,"Recurrent, Metastatic",NaN,No Metastasis,Metastasis,...,No,No,No,No,NaN,17_439_00067_T2,17_439_00067_WB,yes,,499
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,WGS_12_15-WGS_11_13,participant_21,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,No,No,No,NaN,WGS_12_15,WGS_11_13,no,,31
232,WGS_12_19-WGS_11_17,participant_47,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,No,No,No,NaN,WGS_12_19,WGS_11_17,no,,295
233,WGS_12_31-WGS_11_29,participant_31,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,SV,No,No,NaN,WGS_12_31,WGS_11_29,yes,,327
234,WGS_12_7-WGS_11_5,participant_37,MD_Anderson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,No,No,No,No,NaN,WGS_12_7,WGS_11_5,yes,,283


In [117]:
# Export it
SV_number_added_meta.to_csv('../../../metadata/metadata_v12.tsv', sep='\t')
